# Sentinel-2 · Pechea
Run the cells in order. Change PROJECT_ID with the ID of your Google Cloud project activated for Earth Engine. The last cell starts the exports only if you set PORNESTE_EXPORTUL = True .

In [1]:
%pip -q install earthengine-api geemap
import ee
from IPython.display import display

PROJECT_ID = "pechea-flood-sentinel2"  # Replace with your exact GCP Project ID
if PROJECT_ID == "ID_PROIECT":
    raise ValueError("Fill in PROJECT_ID with your Google Cloud project ID.")

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)


## Functions
The code for scenes, comparison, map, and export is included here, so you can run the notebook directly.

In [2]:
import ee

# Order: West longitude, South latitude, East longitude, North latitude.
PECHEA_BOUNDS = [27.74, 45.58, 27.86, 45.68]

# The last day of each interval is excluded. Dates are in UTC.
BEFORE = ("2024-08-15", "2024-09-14")
DURING = ("2024-09-14", "2024-09-19")
AFTER = ("2024-09-19", "2024-10-04")

# UTM 35N uses meters and is suitable for Pechea.
CRS = "EPSG:32635"
PIXEL_SIZE_M = 10


# 1. Search for optical scenes for our selected area and time period.
def cauta_scene(aoi, perioada, max_nori=60):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)  # Search for scenes that intersect with our area
        .filterDate(*perioada)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_nori))  # Filter scenes with high cloud cover
        .select(["B2", "B3", "B4", "B8", "B11", "SCL"])  # Bands
        .sort("system:time_start")
    )


# 2. Display what we found so we can choose a scene by date, clouds, and orbit.
def afiseaza_scene(scene):
    def adauga_detalii(imagine):
        detalii = ee.Dictionary({
            "id": imagine.get("system:index"),
            "data_utc": imagine.date().format("yyyy-MM-dd HH:mm:ss", "UTC"),
            "nori_pct": imagine.get("CLOUDY_PIXEL_PERCENTAGE"),
            "orbita": imagine.get("SENSING_ORBIT_NUMBER"),
            "tile_mgrs": imagine.get("MGRS_TILE")
        })
        return imagine.set("detalii", detalii)

    detalii = scene.map(adauga_detalii).aggregate_array("detalii").getInfo()
    for rand in detalii:
        print(rand)
    return detalii


# 3. Select the event scene and search for scenes with the same geometry.
def alege_scene(aoi, inainte, in_timpul, dupa, *, id_eveniment=None):
    def acopera_zona(scene):
        return scene.filter(
            ee.Filter.contains(leftField=".geo", rightValue=aoi, maxError=1)
        )

    def prima_sau_none(scene):
        if scene.limit(1).size().getInfo() == 0:
            return None
        return ee.Image(scene.first())

    candidate = acopera_zona(cauta_scene(aoi, in_timpul))
    if id_eveniment is not None:
        candidate = candidate.filter(ee.Filter.eq("system:index", id_eveniment))
    else:
        candidate = candidate.sort("CLOUDY_PIXEL_PERCENTAGE")

    eveniment = prima_sau_none(candidate)
    if eveniment is None:
        raise ValueError("No event scene found covering the selected area.")

    def aceeasi_geometrie(perioada):
        return (
            acopera_zona(cauta_scene(aoi, perioada))
            .filter(ee.Filter.eq(
                "SENSING_ORBIT_NUMBER", eveniment.get("SENSING_ORBIT_NUMBER")
            ))
            .filter(ee.Filter.eq(
                "MGRS_TILE", eveniment.get("MGRS_TILE")
            ))
        )

    return {
        "before": prima_sau_none(aceeasi_geometrie(inainte).sort("CLOUDY_PIXEL_PERCENTAGE")),
        "during": eveniment,
        "after": prima_sau_none(aceeasi_geometrie(dupa).sort("CLOUDY_PIXEL_PERCENTAGE")),
    }


# 4. Cloud masking and RGB/False Color extraction
def masca_nori_s2(imagine):
    scl = imagine.select('SCL')
    masca = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return imagine.updateMask(masca)


def obtine_rgb(imagine, aoi, tip="true_color"):
    if imagine is None:
        return None
    img_curat = masca_nori_s2(imagine)

    if tip == "true_color":
        rgb = img_curat.select(["B4", "B3", "B2"]).divide(10000)
    elif tip == "false_color":
        rgb = img_curat.select(["B8", "B4", "B3"]).divide(10000)

    return rgb.clip(aoi)


# Calculate NDWI and MNDWI water indices
def calculeaza_indici_apa(imagine, aoi, *, pixel_m=PIXEL_SIZE_M, crs=CRS):
    if imagine is None:
        raise ValueError("Lipseste scena pentru perioada aleasa.")

    img_curat = masca_nori_s2(imagine)

    # 1. NDWI (B3: Green, B8: NIR) - 10m
    ndwi = img_curat.normalizedDifference(["B3", "B8"]).rename("NDWI")

    # 2. MNDWI (B3: Green, B11: SWIR1) - 10m resampled
    mndwi = img_curat.normalizedDifference(["B3", "B11"]).rename("MNDWI")

    # 3. Grila comuna UTM 35N la 10m
    grila = [pixel_m, 0, 0, 0, -pixel_m, 0]

    indici = ndwi.addBands(mndwi)
    return indici.clip(aoi).reproject(crs=crs, crsTransform=grila).clip(aoi)


# 5. Subtract values pixel by pixel: new index minus baseline index.
def calculeaza_diferenta(inainte_index, nou_index, nume_banda="diferenta_index"):
    return nou_index.subtract(inainte_index).rename(nume_banda)


# 6. Create the initial mask for newly appeared water.
def detecteaza_inundatia(nou_mndwi, diferenta_mndwi, *, prag_apa=0.1, crestere_minima=0.2):
    if crestere_minima < 0:
        raise ValueError("Cresterea minima se da ca numar pozitiv.")

    apa = nou_mndwi.gt(prag_apa)
    a_crescut = diferenta_mndwi.gt(crestere_minima)
    return apa.And(a_crescut).rename("inundatie")


# 7. Evolutia spatiala
def evolutia_inundatiei(masca_eveniment, masca_dupa):
    return masca_eveniment.add(masca_dupa.multiply(2)).rename("evolutie")


# 8. Afisam straturile in geemap
def afiseaza_harta(aoi, inainte_mndwi, eveniment_mndwi, diferenta_mndwi, masca, evolutie=None, imagine_rgb=None):
    import geemap

    # Set base map to Google Maps or Google Satellite to avoid OSM 403 blocks
    harta = geemap.Map(basemap="HYBRID")  # Options: "HYBRID", "SATELLITE", "ROADMAP", or "TERRAIN"
    harta.centerObject(aoi, 12)

    vis_mndwi = {"min": -0.5, "max": 0.5, "palette": ["brown", "yellow", "cyan", "blue"]}

    if imagine_rgb is not None:
        harta.addLayer(imagine_rgb, {"min": 0, "max": 0.3}, "Sentinel-2 True Color (RGB)", False)

    harta.addLayer(inainte_mndwi, vis_mndwi, "MNDWI inainte", False)
    harta.addLayer(eveniment_mndwi, vis_mndwi, "MNDWI in timpul inundatiei")

    harta.addLayer(
        diferenta_mndwi,
        {"min": -0.5, "max": 0.5, "palette": ["red", "white", "blue"]},
        "Diferenta MNDWI", False
    )

    harta.addLayer(masca.selfMask(), {"palette": ["00e5ff"]}, "Candidati inundati")

    if evolutie is not None:
        harta.addLayer(
            evolutie.selfMask(),
            {"min": 1, "max": 3, "palette": ["orange", "cyan", "darkblue"]},
            "Evolutie: retras / nou / persistent", False
        )

    return harta


# 9. Export GeoTIFF
def exporta_geotiff(imagine, aoi, nume, *, folder="ROSPIN_Pechea",
                     pixel_m=PIXEL_SIZE_M, crs=CRS):
    nodata = -9999
    task = ee.batch.Export.image.toDrive(
        image=imagine.clip(aoi).toFloat().unmask(nodata, sameFootprint=False),
        description=nume,
        fileNamePrefix=nume,
        folder=folder,
        region=aoi,
        crs=crs,
        crsTransform=[pixel_m, 0, 0, 0, -pixel_m, 0],
        maxPixels=100_000_000,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True, "noData": nodata},
    )
    task.start()
    return task


# 10. Export Poligoane GeoJSON
def exporta_poligoane(masca, aoi, nume, *, folder="ROSPIN_Pechea"):
    poligoane = masca.selfMask().toInt().reduceToVectors(
        geometry=aoi,
        crs=CRS,
        crsTransform=[PIXEL_SIZE_M, 0, 0, 0, -PIXEL_SIZE_M, 0],
        geometryType="polygon",
        eightConnected=False,
        labelProperty="inundatie",
        maxPixels=100_000_000,
        geometryInNativeProjection=False,
    )
    task = ee.batch.Export.table.toDrive(
        collection=poligoane,
        description=nume,
        folder=folder,
        fileNamePrefix=nume,
        fileFormat="GeoJSON",
    )
    task.start()
    return task


# 11. Metadate Scene
def detalii_scena(imagine):
    if imagine is None:
        return None
    return ee.Dictionary({
        "id": imagine.get("system:index"),
        "acquired_at_utc": imagine.date().format("yyyy-MM-dd'T'HH:mm:ss'Z'", "UTC"),
        "orbit_number": imagine.get("SENSING_ORBIT_NUMBER"),
        "mgrs_tile": imagine.get("MGRS_TILE"),
        "cloud_cover_pct": imagine.get("CLOUDY_PIXEL_PERCENTAGE"),
    }).getInfo()

## Verifica scenele disponibile
Verifica ora UTC si copiaza ID-ul scenei dorite din lista. Daca lasi ID-ul gol, folosim prima scena care acopera zona de lucru.

In [3]:
aoi = ee.Geometry.Rectangle(PECHEA_BOUNDS)

# Căutăm scenele și le sortăm după acoperirea de nori (cele mai senine primele)
scene_eveniment = cauta_scene(aoi, DURING).sort("CLOUDY_PIXEL_PERCENTAGE")

# Afișăm metadatele scenelor găsite
candidate = afiseaza_scene(scene_eveniment)

{'data_utc': '2024-09-18 09:07:42', 'id': '20240918T085549_20240918T085938_T35TNL', 'nori_pct': 1.502101, 'orbita': 7, 'tile_mgrs': '35TNL'}
{'data_utc': '2024-09-16 09:17:44', 'id': '20240916T090621_20240916T091733_T35TNL', 'nori_pct': 15.711075, 'orbita': 50, 'tile_mgrs': '35TNL'}


In [4]:


aoi = ee.Geometry.Rectangle(PECHEA_BOUNDS)

# 1. Alegem automat cele mai curate scene pentru cele 3 perioade
scene = alege_scene(aoi, BEFORE, DURING, AFTER, id_eveniment=None)

print("--- Scene selectate ---")
for perioada, imagine in scene.items():
    print(perioada, detalii_scena(imagine))

if scene["before"] is None:
    raise ValueError("Nu avem o scena dinainte comparabila. Verifica intervalul BEFORE.")

# 2. Calculam indicii acvatici NDWI (10m) si MNDWI (10m resampled)
inainte_indici = calculeaza_indici_apa(scene["before"], aoi)
eveniment_indici = calculeaza_indici_apa(scene["during"], aoi)

inainte_mndwi = inainte_indici.select("MNDWI")
eveniment_mndwi = eveniment_indici.select("MNDWI")

inainte_ndwi = inainte_indici.select("NDWI")
eveniment_ndwi = eveniment_indici.select("NDWI")

# Calculate differences for both water indices
diferenta_mndwi = calculeaza_diferenta(inainte_mndwi, eveniment_mndwi, "diferenta_mndwi")
diferenta_ndwi = calculeaza_diferenta(inainte_ndwi, eveniment_ndwi, "diferenta_ndwi")
praguri = {"prag_apa": 0.1, "crestere_minima": 0.2}
masca = detecteaza_inundatia(eveniment_mndwi, diferenta_mndwi, **praguri)

# 4. Generam stratul True Color (RGB) pentru inspectie vizuala
rgb_eveniment = obtine_rgb(scene["during"], aoi, tip="true_color")

# Pregatim imaginile raster pentru exportul GeoTIFF
imagini = {
    "before_ndwi": inainte_ndwi,
    "during_ndwi": eveniment_ndwi,
    "change_ndwi": diferenta_ndwi,
    "before_mndwi": inainte_mndwi,
    "during_mndwi": eveniment_mndwi,
    "change_mndwi": diferenta_mndwi,
    "flood_mask": masca,
}

# 5. Analizam evolutia spatiala daca exista scena AFTER
evolutie = None
if scene["after"] is not None:
    dupa_indici = calculeaza_indici_apa(scene["after"], aoi)
    dupa_mndwi = dupa_indici.select("MNDWI")

    masca_dupa = detecteaza_inundatia(
        dupa_mndwi, calculeaza_diferenta(inainte_mndwi, dupa_mndwi), **praguri
    )
    evolutie = evolutia_inundatiei(masca, masca_dupa)

    imagini.update(
        after_ndwi=dupa_indici.select("NDWI"),
        after_mndwi=dupa_mndwi,
        flood_evolution=evolutie
    )

# 6. Afisam harta interactiva
display(afiseaza_harta(
    aoi,
    inainte_mndwi,
    eveniment_mndwi,
    diferenta_mndwi,
    masca,
    evolutie=evolutie,
    imagine_rgb=rgb_eveniment
))


--- Scene selectate ---
before {'acquired_at_utc': '2024-08-24T09:07:44Z', 'cloud_cover_pct': 0.003845, 'id': '20240824T085551_20240824T090537_T35TNL', 'mgrs_tile': '35TNL', 'orbit_number': 7}
during {'acquired_at_utc': '2024-09-18T09:07:42Z', 'cloud_cover_pct': 1.502101, 'id': '20240918T085549_20240918T085938_T35TNL', 'mgrs_tile': '35TNL', 'orbit_number': 7}
after {'acquired_at_utc': '2024-09-23T09:07:48Z', 'cloud_cover_pct': 0.173936, 'id': '20240923T085711_20240923T090306_T35TNL', 'mgrs_tile': '35TNL', 'orbit_number': 7}


Map(center=[45.63000083631853, 27.7999999999984], controls=(WidgetControl(options=['position', 'transparent_bg…

## Export pentru aplicatie
Dupa ce verifici harta, seteaza `PORNESTE_EXPORTUL = True` si ruleaza celula o data. Earth Engine scrie rezultatele in Google Drive, in `ROSPIN_Pechea`. `summary.json` se descarca din Colab cu celula de la final.

In [5]:
import json
from pathlib import Path

PORNESTE_EXPORTUL = True
if PORNESTE_EXPORTUL:
    sumar = {
        "event_id": "pechea",
        "name": "Pechea, Galati",
        "source": "COPERNICUS/S2_SR_HARMONIZED",
        "index_type": "MNDWI",
        "aoi_bbox_wgs84": PECHEA_BOUNDS,
        "pixel_size_m": PIXEL_SIZE_M,
        "crs": CRS,
        "thresholds": praguri,
        "scenes": {perioada: detalii_scena(imagine) for perioada, imagine in scene.items()},
        "files": {**{nume: f"{nume}.tif" for nume in imagini},
                  "flood_extent": "flood_extent.geojson"},
        "mask_legend": {"0": "nedetectat", "1": "candidat inundat", "-9999": "fara date (nori/umbre/lipsa)"},
    }
    taskuri = {nume: exporta_geotiff(imagine, aoi, nume)
               for nume, imagine in imagini.items()}
    taskuri["flood_extent"] = exporta_poligoane(masca, aoi, "flood_extent")
    Path("summary.json").write_text(json.dumps(sumar, indent=2), encoding="utf-8")
    print("Exporturi pornite. Verifica starea lor in celula urmatoare.")
else:
    print("Exportul e oprit. Seteaza PORNESTE_EXPORTUL = True cand harta e gata.")


Exporturi pornite. Verifica starea lor in celula urmatoare.


In [6]:
# Ruleaza din nou celula asta pentru starea exporturilor, fara sa le repornesti.
for nume, task in globals().get("taskuri", {}).items():
    status = task.status()
    print(nume, status.get("state"), status.get("error_message", ""))


before_ndwi RUNNING 
during_ndwi READY 
change_ndwi READY 
before_mndwi READY 
during_mndwi READY 
change_mndwi READY 
flood_mask READY 
after_ndwi READY 
after_mndwi READY 
flood_evolution READY 
flood_extent READY 


In [7]:
from google.colab import files
from pathlib import Path

if Path("summary.json").exists():
    files.download("summary.json")
else:
    print("Porneste intai exportul ca sa se genereze summary.json.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>